In [1]:
!pip install mediapipe
!pip install pyautogui
!pip install pycaw

In [2]:
import cv2
import mediapipe as mp
import time
import pyautogui as auto
import pycaw as audio
from pycaw.pycaw import AudioUtilities

In [3]:
#MediaPipe Methods
BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

device = AudioUtilities.GetSpeakers()
volume = device.EndpointVolume

countFist = 0
countOne = 0
countTwo = 0
countThree = 0 
countFour = 0

countMouseTrue = 0
countMouseFalse = 0
mouseMode = False

In [4]:
def print_result(result: mp.tasks.vision.GestureRecognizerResult, output_image: mp.Image, timestamp_ms: int):  
    global countFist
    global countOne
    global countTwo
    global countThree
    global countFour
    global countMouseTrue
    global countMouseFalse
    global mouseMode
    
    if result.gestures:
        # result.gestures is a list of top recognized gestures for each hand
        for gesture in result.gestures:      
            #print(f"Gesture: {gesture[0].category_name} ({round(gesture[0].score, 2)})")
            
            if(gesture[0].category_name == 'palm' and round(gesture[0].score, 2) > .9 and mouseMode == False):             
                countMouseTrue = countMouseTrue + 1
                if(countMouseTrue == 50):
                    mouseMode = True
                    print("V Mouse Enabled")
                    countMouseTrue = 0
                    
            if(gesture[0].category_name == 'palm' and round(gesture[0].score, 2) > .9 and mouseMode == True):             
                countMouseFalse = countMouseFalse + 1
                if(countMouseFalse == 50):
                   mouseMode = False
                   print("V Mouse Disabled")
                   countMouseFalse = 0 
                
            if(gesture[0].category_name == 'one' and round(gesture[0].score, 2) > .9):
                if(mouseMode == True):     
                  countOne = countOne + 1
                  if(countOne == 5):
                     auto.move(-30, 0, .1)
                     countOne = 0
                
                if(mouseMode == False):
                   deviceVolume = volume.GetMasterVolumeLevel()
                   increaseVolume = min(0.0, deviceVolume + .5)
                   volume.SetMasterVolumeLevel(increaseVolume, None)  
            
            if(gesture[0].category_name == 'two' and round(gesture[0].score, 2) > .7):
                if(mouseMode == True):     
                     countTwo = countTwo + 1
                     if(countTwo == 5):
                        auto.move(30, 0, 0.1)
                        countTwo = 0

                if(mouseMode == False):     
                    deviceVolume = volume.GetMasterVolumeLevel()
                    decreaseVolume = min(0.0, deviceVolume - .5)
                    volume.SetMasterVolumeLevel(decreaseVolume, None)
                
            if(gesture[0].category_name == 'three' and round(gesture[0].score, 2) > .8):
                if(mouseMode == True):
                 
                   countThree = countThree + 1
                   if(countThree == 5):
                     auto.move(0, -30, 0.1)
                     countThree = 0
                    
                if(mouseMode == False): 
                 
                    countThree = countThree + 1
                    if(countThree == 5):
                       auto.press('s')
                       countThree = 0
                     
            if(gesture[0].category_name == 'four' and round(gesture[0].score, 2) > .9):         
                if(mouseMode == True): 
                 
                 countFour = countFour + 1
                 if(countFour == 5):
                     auto.move(0, 30, 0.1)
                     countFour = 0
                
                if(mouseMode == False): 
                  
                  countFour = countFour + 1
                  if(countFour == 5):
                     auto.press('backspace')
                     countFour = 0  
                      
            if(gesture[0].category_name == 'fist' and round(gesture[0].score, 2) > .9):
                countFist = countFist + 1
                if(countFist == 10):  
                    auto.click()
                    countFist = 0

In [5]:
options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path='Puffy.task'),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=print_result
)

# 4. Start webcam and process frames
cap = cv2.VideoCapture(0)

with GestureRecognizer.create_from_options(options) as recognizer:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        # Convert the BGR frame to RGB and then to MediaPipe Image format
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # Send live image for processing with a unique timestamp
        frame_timestamp_ms = int(time.time() * 500)
        recognizer.recognize_async(mp_image, frame_timestamp_ms)

        # Display the feed (standard OpenCV window)
        cv2.imshow('Gesture Recognition', frame)
    
        if cv2.waitKey(1) & 0xFF == ord('q'):
                break

cap.release()
cv2.destroyAllWindows()